In [238]:
import matplotlib
matplotlib.use('Qt5Agg')  # Important for VSCode interactivity

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import scanpy as sc
import os
import scipy.sparse as sp
import anndata as ad
import pandas as pd
from scipy.spatial import cKDTree
from tqdm import tqdm
from skimage.measure import find_contours

In [239]:
# Load and visualize the m/z heatmap from AnnData
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/lipdid dispersion paper/Hearts/h5ad_data_overlaid/h_5_2_overlaid.h5ad"
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

Loaded AnnData from: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/lipdid dispersion paper/Hearts/h5ad_data_overlaid/h_5_2_overlaid.h5ad


In [240]:
batch = adata.obs['batch'][0] 
sample = adata.obs['sample'][0]
adata

/var/folders/0n/w0dydx896p7b5chql0_cv6hr0000gn/T/ipykernel_48345/3043734248.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  batch = adata.obs['batch'][0]
/var/folders/0n/w0dydx896p7b5chql0_cv6hr0000gn/T/ipykernel_48345/3043734248.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  sample = adata.obs['sample'][0]


AnnData object with n_obs × n_vars = 8391 × 1000
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status', 'tissue'
    var: 'mzs'

In [241]:

def tic_normalize(adata, inplace = True, layer_name = None):
    """
    Perform Total Ion Current (TIC) normalization on an AnnData object.
    
    Parameters:
    - adata (AnnData): The input AnnData object with intensity matrix in `.X`.
    - inplace (bool): If True, overwrite `adata.X`. If False, store result in `adata.layers[layer_name]`.
    - layer_name (str): Name of the layer to store normalized matrix. Required if `inplace=False`.

    Returns:
    - None. The AnnData object is modified in-place.
    """

    if not inplace and not layer_name:
        raise ValueError("You must specify 'layer_name' when inplace=False.")

    # Compute TIC per observation
    tic = adata.X.sum(axis=1)
    tic = np.asarray(tic).flatten()
    tic[tic == 0] = 1.0  # Avoid division by zero

    # Normalize
    if sp.issparse(adata.X):
        normalized = adata.X.multiply(1 / tic[:, np.newaxis])
    else:
        normalized = adata.X / tic[:, np.newaxis]

    # Store result
    if inplace:
        adata.layers["raw"] = adata.X.copy()  # Store raw values before normalization
        adata.X = normalized
    else:
        adata.layers[layer_name] = normalized

    # Save TIC to .obs
    adata.obs["tic"] = tic

    print(f"✅ TIC normalization complete. {'Updated adata.X' if inplace else f'Stored in layer: {layer_name}'}")
    

In [242]:
tic_normalize(adata, inplace=True)

✅ TIC normalization complete. Updated adata.X


In [243]:
def filter_mz_by_tissue(adata, tissue_key="tissue", tissue_val=1, non_tissue_val=0, foldchange=2.0, layer=None, normalize_tic=True):
    # 1. Extract raw data
    data = adata.layers[layer] if layer is not None else adata.X

    # 2. TIC normalization per pixel
    if normalize_tic:
        # Sum across features for each observation (axis=1)
        tic = np.asarray(data.sum(axis=1)).ravel()
        # Avoid division by zero
        tic[tic == 0] = 1.0
        # Normalize in place (if sparse, convert to array)
        data = data / tic[:, None]

    # 3. Define tissue masks
    mask_tissue = adata.obs[tissue_key] == tissue_val
    mask_non    = adata.obs[tissue_key] == non_tissue_val

    # 4. Compute mean normalized intensity per feature
    mean_tissue    = np.asarray(data[mask_tissue].mean(axis=0)).ravel()
    mean_nontissue = np.asarray(data[mask_non].mean(axis=0)).ravel()

    # 5. Identify features to remove
    to_remove = mean_tissue < foldchange * mean_nontissue

    # 6. Print removed m/z names
    mz_to_remove = adata.var_names[to_remove]
    print(f"Dropping {len(mz_to_remove)} m/z features:", mz_to_remove.values)

    # 7. Subset and return
    return adata[:, ~to_remove].copy()

In [244]:
# I usually choose a foldchange of 3, but you can adjust this based on your data
foldchange = 3

# Filter m/z features based on tissue mask
adata = filter_mz_by_tissue(adata, tissue_key="tissue", tissue_val=1, non_tissue_val=0, foldchange=foldchange, layer=None, normalize_tic=False)

Dropping 854 m/z features: ['251.9215' '255.0288' '256.0408' '257.0479' '258.0499' '259.0608'
 '260.0386' '261.0393' '261.2228' '262.0491' '262.2258' '263.0537'
 '263.2379' '264.0602' '264.2448' '265.2536' '269.0430' '271.0232'
 '272.0305' '273.0396' '274.0433' '274.2747' '275.0562' '276.0636'
 '277.0657' '279.0387' '279.2357' '285.0367' '285.2358' '286.0401'
 '287.0453' '288.0300' '289.0386' '290.0416' '291.0538' '292.0602'
 '293.0609' '295.0226' '296.0284' '297.0358' '298.0451' '303.0180'
 '303.1168' '305.0354' '305.1270' '306.0352' '307.0367' '307.1208'
 '308.0550' '308.1164' '309.0444' '309.1365' '310.1353' '311.0125'
 '311.1511' '311.2589' '312.0220' '313.0332' '313.2727' '314.0382'
 '314.2781' '315.0449' '316.0454' '316.0998' '317.0475' '317.1020'
 '318.1135' '319.0409' '326.3823' '329.0223' '330.0209' '331.0291'
 '331.1085' '332.0307' '332.0944' '332.3332' '333.0180' '333.1216'
 '333.3368' '334.0227' '334.1105' '335.0129' '335.1328' '335.2607'
 '336.0207' '336.1327' '337.2784' '

In [245]:
adata

AnnData object with n_obs × n_vars = 8391 × 146
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status', 'tissue', 'tic'
    var: 'mzs'
    layers: 'raw'

In [246]:
def compute_center_of_mass(x_coords, y_coords, intensities):
    total_intensity = np.sum(intensities)
    if total_intensity == 0:
        return np.nan, np.nan
    x_center = np.sum(x_coords * intensities) / total_intensity
    y_center = np.sum(y_coords * intensities) / total_intensity
    return x_center, y_center


def analyze_metrics(adata, export_border_points=False):
    num_mz = adata.shape[1]
    mz_values = adata.var["mzs"].values

    all_border_data = []
    rows = []

    tissue_centroid_x = adata.obs.loc[adata.obs["tissue"] == True, "x"].mean()
    tissue_centroid_y = adata.obs.loc[adata.obs["tissue"] == True, "y"].mean()

    for mz_index in tqdm(range(num_mz), desc="Processing m/z features"):
        row = {"mz": mz_values[mz_index]}

        # Global center of mass and distance to tissue centroid
        all_x = adata.obs["x"].values
        all_y = adata.obs["y"].values
        all_intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index].flatten()
        x_com, y_com = compute_center_of_mass(all_x, all_y, all_intensities)

        if not np.isnan(x_com) and not np.isnan(y_com):
            dist_to_tissue_centroid = np.sqrt((x_com - tissue_centroid_x) ** 2 + (y_com - tissue_centroid_y) ** 2)
        else:
            dist_to_tissue_centroid = np.nan

        row["com_distance_to_tissue"] = dist_to_tissue_centroid

        # Background distance to tissue border for nonzero-intensity pixels
        bg_subset = adata[adata.obs["tissue"] == False]
        tissue_subset = adata[adata.obs["tissue"] == True]

        bg_x = bg_subset.obs["x"].values
        bg_y = bg_subset.obs["y"].values
        bg_intensities = bg_subset.X[:, mz_index].toarray().flatten() if hasattr(bg_subset.X, "toarray") else bg_subset.X[:, mz_index].flatten()

        tissue_x = tissue_subset.obs["x"].values
        tissue_y = tissue_subset.obs["y"].values

        if len(bg_intensities) > 0 and len(tissue_x) > 0:
            valid_mask = bg_intensities > 0
            valid_bg_coords = np.vstack([bg_x[valid_mask], bg_y[valid_mask]]).T
            tissue_coords = np.vstack([tissue_x, tissue_y]).T

            if valid_bg_coords.shape[0] > 0:
                tree = cKDTree(tissue_coords)
                distances, indices = tree.query(valid_bg_coords)

                nearest_tissue_points = tissue_coords[indices]

                row["bg_dist_max"] = np.max(distances)
                row["bg_dist_mean"] = np.mean(distances)
                row["bg_nonzero_area"] = valid_bg_coords.shape[0]

                if export_border_points:
                    for i in range(len(valid_bg_coords)):
                        all_border_data.append({
                            "mz": mz_values[mz_index],
                            "bg_x": valid_bg_coords[i, 0],
                            "bg_y": valid_bg_coords[i, 1],
                            "tissue_x": nearest_tissue_points[i, 0],
                            "tissue_y": nearest_tissue_points[i, 1],
                            "distance": distances[i]
                        })

            else:
                row["bg_dist_max"] = np.nan
                row["bg_dist_mean"] = np.nan
                row["bg_nonzero_area"] = 0
        else:
            row["bg_dist_max"] = np.nan
            row["bg_dist_mean"] = np.nan
            row["bg_nonzero_area"] = 0

        # Weighted centroids
        tissue_intensities = tissue_subset.X[:, mz_index].toarray().flatten() if hasattr(tissue_subset.X, "toarray") else tissue_subset.X[:, mz_index].flatten()
        tissue_centroid_x_weighted, tissue_centroid_y_weighted = compute_center_of_mass(tissue_x, tissue_y, tissue_intensities)
        row["weighted_com_x_tissue"] = tissue_centroid_x_weighted
        row["weighted_com_y_tissue"] = tissue_centroid_y_weighted
        row["weighted_com_x_all"] = x_com
        row["weighted_com_y_all"] = y_com

        rows.append(row)

    df_metrics = pd.DataFrame(rows)

    if export_border_points:
        df_border = pd.DataFrame(all_border_data)
        return df_metrics, df_border
    else:
        return df_metrics

In [247]:
pre_df_metrics, pre_df_border = analyze_metrics(adata, export_border_points=True)

Processing m/z features: 100%|██████████| 146/146 [00:01<00:00, 113.62it/s]


In [248]:
def mask_low_background_intensities(adata, fold_change=0.2):
    """
    Set background pixel intensities (tissue==0) to 0 if they are below fold_change * max tissue intensity per m/z.
    
    Parameters:
        adata : AnnData
            An AnnData object where .X is (n_pixels x n_mz) and .obs["tissue"] exists.
        fold_change : float
            The threshold multiplier to compare background intensity to max tissue intensity.
    """
    X = adata.X
    tissue_mask = adata.obs["tissue"] == 1
    background_mask = ~tissue_mask

    if sp.issparse(X):
        X = X.tocsr()

    # Compute max intensity of each m/z in tissue pixels
    max_tissue_intensity = X[tissue_mask.values].max(axis=0).A1 if sp.issparse(X) else X[tissue_mask.values].max(axis=0)

    # Define threshold per m/z
    thresholds = fold_change * max_tissue_intensity

    # Modify background pixels
    bg_indices = np.where(background_mask)[0]

    for i in bg_indices:
        row = X[i]
        if sp.issparse(X):
            row = row.tocoo()
            mask = row.data <= thresholds[row.col]
            row.data[mask] = 0
            X[i] = sp.csr_matrix((row.data, (np.zeros_like(row.col), row.col)), shape=(1, X.shape[1]))
        else:
            mask = row <= thresholds
            row[mask] = 0
            X[i] = row

    adata.X = X
    return adata

In [249]:
min_background_intensity = 0.1
adata = mask_low_background_intensities(adata, fold_change=min_background_intensity)

In [250]:

def compute_center_of_mass(x_coords, y_coords, intensities):
    total_intensity = np.sum(intensities)
    if total_intensity == 0:
        return np.nan, np.nan
    x_center = np.sum(x_coords * intensities) / total_intensity
    y_center = np.sum(y_coords * intensities) / total_intensity
    return x_center, y_center


def analyze_metrics(adata, export_border_points=False):
    num_mz = adata.shape[1]
    mz_values = adata.var["mzs"].values

    all_border_data = []
    rows = []

    for mz_index in tqdm(range(num_mz), desc="Processing m/z features"):

        # Initialize row for current m/z
        row = {"mz": mz_values[mz_index]}

        # Extract x, y, and intensity for current mz
        all_x = adata.obs["x"].values
        all_y = adata.obs["y"].values
        all_intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index].flatten()

        # Compute global center of mass
        x_com, y_com = compute_center_of_mass(all_x, all_y, all_intensities)

        # Only tissue pixels
        tissue_mask = adata.obs["tissue"].values == True
        tissue_x = all_x[tissue_mask]
        tissue_y = all_y[tissue_mask]
        tissue_intensities = all_intensities[tissue_mask]

        # Compute weighted tissue centroid (center of mass within tissue)
        x_com_tissue, y_com_tissue = compute_center_of_mass(tissue_x, tissue_y, tissue_intensities)

        # Compute distance from global COM to tissue COM
        if not np.isnan(x_com) and not np.isnan(y_com):
            dist_to_tissue_centroid = np.sqrt((x_com - x_com_tissue) ** 2 + (y_com - y_com_tissue) ** 2)
        else:
            dist_to_tissue_centroid = np.nan

        row["com_distance_to_tissue"] = dist_to_tissue_centroid

        # Background distance to tissue border for nonzero-intensity pixels
        bg_subset = adata[adata.obs["tissue"] == False]
        tissue_subset = adata[adata.obs["tissue"] == True]

        bg_x = bg_subset.obs["x"].values
        bg_y = bg_subset.obs["y"].values
        bg_intensities = bg_subset.X[:, mz_index].toarray().flatten() if hasattr(bg_subset.X, "toarray") else bg_subset.X[:, mz_index].flatten()

        tissue_x = tissue_subset.obs["x"].values
        tissue_y = tissue_subset.obs["y"].values
        tissue_intensities = tissue_subset.X[:, mz_index].toarray().flatten() if hasattr(tissue_subset.X, "toarray") else tissue_subset.X[:, mz_index].flatten()

        # Add mean intensities
        row["mean_intensity_tissue"] = np.mean(tissue_intensities) if tissue_intensities.size > 0 else np.nan
        row["mean_intensity_bg"] = np.mean(bg_intensities) if bg_intensities.size > 0 else np.nan

        if len(bg_intensities) > 0 and len(tissue_x) > 0:
            valid_mask = bg_intensities > 0
            valid_bg_coords = np.vstack([bg_x[valid_mask], bg_y[valid_mask]]).T
            tissue_coords = np.vstack([tissue_x, tissue_y]).T

            if valid_bg_coords.shape[0] > 0:
                tree = cKDTree(tissue_coords)
                distances, indices = tree.query(valid_bg_coords)

                nearest_tissue_points = tissue_coords[indices]

                row["bg_dist_max"] = np.max(distances)
                row["bg_dist_mean"] = np.mean(distances)
                row["bg_nonzero_area"] = valid_bg_coords.shape[0]

                if export_border_points:
                    for i in range(len(valid_bg_coords)):
                        all_border_data.append({
                            "mz": mz_values[mz_index],
                            "bg_x": valid_bg_coords[i, 0],
                            "bg_y": valid_bg_coords[i, 1],
                            "tissue_x": nearest_tissue_points[i, 0],
                            "tissue_y": nearest_tissue_points[i, 1],
                            "distance": distances[i]
                        })

            else:
                row["bg_dist_max"] = np.nan
                row["bg_dist_mean"] = np.nan
                row["bg_nonzero_area"] = 0
        else:
            row["bg_dist_max"] = np.nan
            row["bg_dist_mean"] = np.nan
            row["bg_nonzero_area"] = 0

        # Weighted centroids
        tissue_intensities = tissue_subset.X[:, mz_index].toarray().flatten() if hasattr(tissue_subset.X, "toarray") else tissue_subset.X[:, mz_index].flatten()
        tissue_centroid_x_weighted, tissue_centroid_y_weighted = compute_center_of_mass(tissue_x, tissue_y, tissue_intensities)
        row["weighted_com_x_tissue"] = tissue_centroid_x_weighted
        row["weighted_com_y_tissue"] = tissue_centroid_y_weighted
        row["weighted_com_x_all"] = x_com
        row["weighted_com_y_all"] = y_com

        rows.append(row)

    df_metrics = pd.DataFrame(rows)

    if export_border_points:
        df_border = pd.DataFrame(all_border_data)
        return df_metrics, df_border
    else:
        return df_metrics

In [251]:
def visualize_distance_map(df_border, mz_value, n_arrows=100):
    subset = df_border[df_border["mz"] == mz_value]
    if subset.empty:
        print(f"No border data found for m/z {mz_value}")
        return

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(subset["bg_x"], subset["bg_y"], c="blue", s=10, label="Background Pixels")
    ax.scatter(subset["tissue_x"], subset["tissue_y"], c="red", s=10, label="Nearest Tissue Border")

    # Draw arrows from bg pixel to nearest tissue
    sample = subset.sample(n=min(n_arrows, len(subset)))
    for _, row in sample.iterrows():
        ax.arrow(row["bg_x"], row["bg_y"],
                 row["tissue_x"] - row["bg_x"],
                 row["tissue_y"] - row["bg_y"],
                 color="gray", alpha=0.5, head_width=1)

    ax.set_title(f"Nearest Tissue Border Points for m/z {mz_value}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend()
    ax.axis("equal")
    plt.tight_layout()
    plt.show()

In [252]:
df_metrics, df_border = analyze_metrics(adata, export_border_points=True)

Processing m/z features: 100%|██████████| 146/146 [00:00<00:00, 194.22it/s]


In [253]:
# Visualize arrows for m/z=885.55 (example)
visualize_distance_map(df_border, mz_value=772.5231)


No border data found for m/z 772.5231


In [254]:

def visualize_mz_feature(adata, df_metrics, df_border, mz_value, cmap="hot"):
    # Find index of target m/z
    mz_idx = np.argmin(np.abs(adata.var["mzs"].values - mz_value))

    # Get coordinates and intensities
    x = adata.obs["x"].values
    y = adata.obs["y"].values
    intensities = adata.X[:, mz_idx].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_idx].flatten()

    tissue_mask = adata.obs["tissue"].values

    # Extract COMs and border points
    mz_metrics = df_metrics[df_metrics["mz"] == mz_value].iloc[0]
    com_all = (mz_metrics["weighted_com_x_all"], mz_metrics["weighted_com_y_all"])
    com_tissue = (mz_metrics["weighted_com_x_tissue"], mz_metrics["weighted_com_y_tissue"])

    df_mz_border = df_border[df_border["mz"] == mz_value]
    max_row = df_mz_border.loc[df_mz_border["distance"].idxmax()]
    bg_point = (max_row["bg_x"], max_row["bg_y"])
    nearest_tissue_point = (max_row["tissue_x"], max_row["tissue_y"])

    # Create 2D grid (image) for tissue mask
    grid_x = np.unique(x)
    grid_y = np.unique(y)
    x_to_idx = {v: i for i, v in enumerate(grid_x)}
    y_to_idx = {v: i for i, v in enumerate(grid_y)}
    mask_2d = np.zeros((len(grid_y), len(grid_x)), dtype=bool)

    for xi, yi, is_tissue in zip(x, y, tissue_mask):
        if is_tissue:
            mask_2d[y_to_idx[yi], x_to_idx[xi]] = True

    contours = find_contours(mask_2d.astype(float), level=0.5)

    # Create figure
    plt.figure(figsize=(10, 8))

    # Plot heatmap with square pixels
    plt.scatter(x, y, c=intensities, s=11, marker='s', cmap=cmap)
    plt.colorbar(label=f"Intensity @ m/z {mz_value:.2f}")

    # Plot center of mass
    plt.scatter(*com_all, color="cyan", s=150, marker="*", label="COM (all)")
    plt.scatter(*com_tissue, color="lime", s=150, marker="*", label="COM (tissue)")

    # Max distance point and nearest tissue point
    plt.plot([bg_point[0], nearest_tissue_point[0]], [bg_point[1], nearest_tissue_point[1]], 
             color="orange", linestyle="--", label="Max dist to tissue")
    plt.scatter(*bg_point, color="red", s=80, edgecolor="black", label="Furthest BG pt")
    plt.scatter(*nearest_tissue_point, color="green", s=80, edgecolor="black", label="Nearest tissue pt")

    # Plot tissue margin line
    for contour in contours:
        contour_x = grid_x[(contour[:, 1]).astype(int)]
        contour_y = grid_y[(contour[:, 0]).astype(int)]
        plt.plot(contour_x, contour_y, color="red", linewidth=2, label="Tissue border")

    plt.legend()
    plt.title(f"m/z {mz_value:.2f} Heatmap with COMs and Furthest BG Point")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.gca().set_aspect('equal')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

In [255]:
custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),   # white
    (0.00000001, "#000000"),  # black
    (0.10, "#000080"),  # navy
    (0.15, "#0000FF"),  # blue
    (0.30, "#8000FF"),  # purple-ish
    (0.45, "#FF0000"),  # red
    (0.60, "#FF8000"),  # orange
    (0.75, "#FFFF00"),  # yellow
    (1.0, "#FFFFFF")   # white
])
#custom_cmap = "binary"
#visualize_mz_feature(adata, df_metrics, df_border, mz_value=800.5231, cmap=custom_cmap)

In [256]:
def visualize_mz_feature(adata, df_metrics, df_border, mz_value, cmap="hot", save_path=None):
    from skimage.measure import find_contours
    import matplotlib.pyplot as plt
    import numpy as np

    # Find index of target m/z
    mz_idx = np.argmin(np.abs(adata.var["mzs"].values - mz_value))

    # Get coordinates and intensities
    x = adata.obs["x"].values
    y = adata.obs["y"].values
    intensities = adata.X[:, mz_idx].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_idx].flatten()

    tissue_mask = adata.obs["tissue"].values

    # Extract COMs and border points
    mz_metrics = df_metrics[df_metrics["mz"] == mz_value].iloc[0]
    com_all = (mz_metrics["weighted_com_x_all"], mz_metrics["weighted_com_y_all"])
    com_tissue = (mz_metrics["weighted_com_x_tissue"], mz_metrics["weighted_com_y_tissue"])

    df_mz_border = df_border[df_border["mz"] == mz_value]
    if df_mz_border.empty:
        print(f"⚠️ Skipping m/z {mz_value:.4f} — no matching metric data found.")
    else:
        max_row = df_mz_border.loc[df_mz_border["distance"].idxmax()]
        bg_point = (max_row["bg_x"], max_row["bg_y"])
        nearest_tissue_point = (max_row["tissue_x"], max_row["tissue_y"])

    # Create 2D grid (image) for tissue mask
    grid_x = np.unique(x)
    grid_y = np.unique(y)
    x_to_idx = {v: i for i, v in enumerate(grid_x)}
    y_to_idx = {v: i for i, v in enumerate(grid_y)}
    mask_2d = np.zeros((len(grid_y), len(grid_x)), dtype=bool)

    for xi, yi, is_tissue in zip(x, y, tissue_mask):
        if is_tissue:
            mask_2d[y_to_idx[yi], x_to_idx[xi]] = True

    contours = find_contours(mask_2d.astype(float), level=0.5)

    # Create figure
    plt.figure(figsize=(10, 8))

    # Plot heatmap
    plt.scatter(x, y, c=intensities, s=12, marker='s', cmap=cmap)
    plt.colorbar(label=f"Intensity @ m/z {mz_value:.2f}")

    # Plot COMs and max distance markers
    plt.scatter(*com_all, color="cyan", s=150, marker="*", label="COM (all)")
    plt.scatter(*com_tissue, color="lime", s=150, marker="*", label="COM (tissue)")
    if not df_mz_border.empty:
        plt.plot([bg_point[0], nearest_tissue_point[0]], [bg_point[1], nearest_tissue_point[1]], 
                color="orange", linestyle="--", label="Max dist to tissue")
        plt.scatter(*bg_point, color="red", s=80, edgecolor="black", label="Furthest BG pt")
        plt.scatter(*nearest_tissue_point, color="green", s=80, edgecolor="black", label="Nearest tissue pt")

    # Plot tissue contour
    for contour in contours:
        contour_x = grid_x[(contour[:, 1]).astype(int)]
        contour_y = grid_y[(contour[:, 0]).astype(int)]
        plt.plot(contour_x, contour_y, color="red", linewidth=2, label="Tissue border")

    plt.legend()
    plt.title(f"m/z {mz_value:.2f} Heatmap with COMs and Furthest BG Point")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.gca().set_aspect('equal')
    plt.gca().invert_yaxis()
    plt.tight_layout()

    # Save or show
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.close()
    else:
        plt.show()


In [257]:
df_metrics

,mz,com_distance_to_tissue,mean_intensity_tissue,mean_intensity_bg,bg_dist_max,bg_dist_mean,bg_nonzero_area,weighted_com_x_tissue,weighted_com_y_tissue,weighted_com_x_all,weighted_com_y_all
0,583.5059,5.004284,0.000492,0.000111,21.633308,7.839905,1736,73.677301,49.183902,69.540562,51.999972
1,651.5319,1.728740,0.002551,0.000203,21.260292,6.933961,616,81.265247,53.769705,80.339486,55.229674
2,652.5354,2.053380,0.001137,0.000107,21.260292,7.374852,697,81.052465,53.568002,80.011147,55.337754
3,653.5507,3.097372,0.000965,0.000202,21.260292,7.281534,1900,77.521630,51.579108,75.728558,54.104698
4,697.4737,1.622865,0.000478,0.000041,21.095023,7.865101,553,75.981579,50.102919,74.426930,49.637345
...,...,...,...,...,...,...,...,...,...,...,...
141,1592.1654,0.583840,0.000279,0.000004,15.033296,2.163715,62,78.584723,51.610185,78.253713,51.129246
142,1593.1761,0.733521,0.000222,0.000004,16.401219,2.870445,81,78.470100,51.643695,78.008944,51.073267
143,1594.1870,1.051342,0.000225,0.000006,16.124515,3.443885,136,78.469487,51.421216,77.824015,50.591344
144,1595.1984,1.262372,0.000154,0.000007,21.095023,4.461706,210,77.518907,50.732532,76.706291,49.766489


In [258]:
def score_mz_features(df_metrics, area_weight=0.95):
    # Normalize metrics
    # Normalize distance metrics
    df_metrics["normalized_bg_dist_mean"] = (
        df_metrics["bg_dist_mean"] - df_metrics["bg_dist_mean"].min()
    ) / (df_metrics["bg_dist_mean"].max() - df_metrics["bg_dist_mean"].min())

    # Normalize area metrics
    df_metrics["normalized_bg_nonzero_area"] = (
        df_metrics["bg_nonzero_area"] - df_metrics["bg_nonzero_area"].min()
    ) / (df_metrics["bg_nonzero_area"].max() - df_metrics["bg_nonzero_area"].min())

    # Calculate dispersion score
    # The dispersion score is a weighted combination of the normalized metrics
    # Higher score means better dispersion (lower distance, higher area)
    # Note: area_weight controls the balance between area and distance
    df_metrics["dispersion_score"] = (
        area_weight * df_metrics["normalized_bg_nonzero_area"]
        + (1 - area_weight) * df_metrics["normalized_bg_dist_mean"]
    )
    
    return df_metrics


In [259]:
area_weight = 0.95
# Score m/z features based on dispersion metrics
# This will add a new column 'dispersion_score' to df_metrics
df_metrics = score_mz_features(df_metrics, area_weight=area_weight)
df_metrics

,mz,com_distance_to_tissue,mean_intensity_tissue,mean_intensity_bg,bg_dist_max,bg_dist_mean,bg_nonzero_area,weighted_com_x_tissue,weighted_com_y_tissue,weighted_com_x_all,weighted_com_y_all,normalized_bg_dist_mean,normalized_bg_nonzero_area,dispersion_score
0,583.5059,5.004284,0.000492,0.000111,21.633308,7.839905,1736,73.677301,49.183902,69.540562,51.999972,0.877425,0.842080,0.843847
1,651.5319,1.728740,0.002551,0.000203,21.260292,6.933961,616,81.265247,53.769705,80.339486,55.229674,0.758075,0.297862,0.320873
2,652.5354,2.053380,0.001137,0.000107,21.260292,7.374852,697,81.052465,53.568002,80.011147,55.337754,0.816158,0.337221,0.361167
3,653.5507,3.097372,0.000965,0.000202,21.260292,7.281534,1900,77.521630,51.579108,75.728558,54.104698,0.803864,0.921769,0.915873
4,697.4737,1.622865,0.000478,0.000041,21.095023,7.865101,553,75.981579,50.102919,74.426930,49.637345,0.880744,0.267250,0.297924
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141,1592.1654,0.583840,0.000279,0.000004,15.033296,2.163715,62,78.584723,51.610185,78.253713,51.129246,0.129636,0.028669,0.033717
142,1593.1761,0.733521,0.000222,0.000004,16.401219,2.870445,81,78.470100,51.643695,78.008944,51.073267,0.222742,0.037901,0.047143
143,1594.1870,1.051342,0.000225,0.000006,16.124515,3.443885,136,78.469487,51.421216,77.824015,50.591344,0.298287,0.064626,0.076309
144,1595.1984,1.262372,0.000154,0.000007,21.095023,4.461706,210,77.518907,50.732532,76.706291,49.766489,0.432376,0.100583,0.117173


In [260]:
df_metrics_filtered = df_metrics[["mz", "dispersion_score", "bg_nonzero_area","normalized_bg_nonzero_area", "bg_dist_mean", "normalized_bg_dist_mean","mean_intensity_tissue", "mean_intensity_bg"]]
df_metrics_filtered.to_csv(f"/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/lipdid dispersion paper/Hearts/results/mz_metrics_{sample}.csv", index=False)

In [261]:
output_folder = f"/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/lipdid dispersion paper/Hearts/results/{sample}_output_{min_background_intensity}_score_{area_weight}"

if os.path.exists(output_folder):
    print(f"⚠️ Warning: Output folder '{output_folder}' already exists. Overwriting.")
else:
    os.makedirs(output_folder, exist_ok=True)

df_sorted = df_metrics.sort_values(by="dispersion_score", ascending=False)
rank_num = 1

# Visualize and save m/z features based on dispersion score
# Loop through sorted m/z values and save visualizations
for mz in df_sorted["mz"]:
    filename = f"{rank_num}_mz_{mz:.4f}.png"
    filepath = os.path.join(output_folder, filename)
    
    custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
        (0.0, "#454545"),   # white
        (0.00000001, "#000000"),  # black
        (0.10, "#000080"),  # navy
        (0.15, "#0000FF"),  # blue
        (0.30, "#8000FF"),  # purple-ish
        (0.45, "#FF0000"),  # red
        (0.60, "#FF8000"),  # orange
        (0.75, "#FFFF00"),  # yellow
        (1.0, "#FFFFFF")   # white
    ])
    
    visualize_mz_feature(adata, df_metrics, df_border, mz_value=mz, cmap=custom_cmap, save_path=filepath)
    print(f"Saved: {filepath}")
    rank_num += 1

Saved: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/lipdid dispersion paper/Hearts/results/5-2_output_0.1_score_0.95/1_mz_747.4973.png
Saved: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/lipdid dispersion paper/Hearts/results/5-2_output_0.1_score_0.95/2_mz_742.5395.png
Saved: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/lipdid dispersion paper/Hearts/results/5-2_output_0.1_score_0.95/3_mz_653.5507.png
Saved: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/lipdid dispersion paper/Hearts/results/5-2_output_0.1_score_0.95/4_mz_853.6340.png
Saved: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/lipdid dispersion paper/Hearts/results/5-2_output_0.1_score_0.95/5_mz_810.5131.png
Saved: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/lipdid dispersion paper/Hearts/results/5-2_output_0.1_score_0.95/6_mz_909.5543.png
Saved: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/lipdid dispersion paper/Hearts/results/5-2_output_0.1_score_0.95/7_mz_583.5059.png
Saved: /Users/amin/Desktop/PhD_The